## Hybrid search, reranking and modern chunking

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mzarnecki/course_llm_agent_apps_with_langchain_and_langgraph/blob/main/notebooks/en/7_4_Hybrid_search_reranking_chunking_en.ipynb)

| | |
|---|---|
| 🎯 **What you'll learn** | Production RAG retrieval: BM25 + embeddings fused with Reciprocal Rank Fusion, cross-encoder reranking, and six chunking strategies including Contextual Retrieval. |
| 🔑 **API keys** | `OPENAI_API_KEY` |
| 💲 **API cost** | very low – embeddings of a few dozen short texts and ~10 short LLM calls |
| ⏱️ **Run time** | ~1–3 min (first run downloads the ~2.3 GB `BAAI/bge-reranker-v2-m3` model) |

**Plan:**
1. Hybrid search – BM25 + embeddings + Reciprocal Rank Fusion (RRF)
2. Reranking with a cross-encoder
3. Modern chunking strategies – fixed, recursive, semantic, structure-aware, parent-child, Contextual Retrieval
4. Everything together – a production-style RAG pipeline

### Install libraries

In [25]:
# Install this notebook's direct dependencies.
# Skipped when pip is missing, e.g. in the `uv sync` environment, which already has them (see README).
import importlib.util
if importlib.util.find_spec("pip"):
    %pip install -qU langchain-openai langchain-community langchain-classic langchain-text-splitters rank-bm25 faiss-cpu sentence-transformers pandas python-dotenv
# Google Colab: add OPENAI_API_KEY in the Secrets panel (🔑) and grant this notebook access.
# Uncomment these lines to load it from Colab Secrets into this runtime:
# from google.colab import userdata
# import os
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
# Locally, put OPENAI_API_KEY=... in a .env file; do not commit the key.

Note: you may need to restart the kernel to use updated packages.


In [26]:
import warnings

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# langchain-community prints a sunset notice on import; its retrievers still work fine
warnings.filterwarnings("ignore", category=DeprecationWarning)
load_dotenv()

llm = ChatOpenAI(model="gpt-6-luna")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

## 1. Hybrid search – BM25 + embeddings

Semantic (vector) search understands **meaning**, but it is surprisingly bad at **exact tokens**: error codes, IDs, contract numbers, product SKUs. To an embedding model `ERR-10532` and `ERR-10533` look almost identical. Lexical search (BM25) is the opposite: it matches exact words, but knows nothing about synonyms.

| Example query | Preferred method |
|---|---|
| "How do I end my agreement?" | Semantic search |
| "Document HRB 80430" | Lexical search |
| "Problems connecting to the database" | Semantic search |
| "Error ERR-10532" | Lexical search |
| "Termination conditions of contract no. 120/2026" | Hybrid search |

Hybrid search runs both and fuses the rankings:

```
                  Query
                    │
         ┌──────────┴──────────┐
         ▼                     ▼
    BM25 Search           Vector Search
         │                     │
         └──────────┬──────────┘
                    ▼
            Rank Fusion (RRF)
                    ▼
              Best documents
```

We don't need Elasticsearch for this – `rank-bm25`, FAISS and OpenAI embeddings are enough.

### Knowledge base

In [27]:
from langchain_core.documents import Document

knowledge_base = {
    "contract-general": "Ending the agreement: either party may terminate the service agreement with 30 days' written notice sent by e-mail or letter.",
    "contract-120": "Contract no. 120/2026 with Nordic Logistics: termination requires 90 days' notice and settlement of all open invoices.",
    "contract-121": "Contract no. 121/2026 with Baltic Foods: termination requires 60 days' notice; the deposit is returned within 14 days.",
    "db-troubleshooting": "If the application cannot reach PostgreSQL, check whether the connection pool is exhausted and whether the firewall allows traffic on port 5432.",
    "db-backup": "Database backups run every night at 2:00 and are kept for 30 days.",
    "refunds": "Refunds for card payments are processed within 5–10 business days after approval.",
    "payment-methods": "We accept Visa, Mastercard, BLIK and bank transfers. American Express is not supported.",
    "password": "To reset your password click 'Forgot password' on the login page and follow the link sent by e-mail.",
}

# real knowledge bases contain many near-identical identifiers
error_messages = [
    "the bank declined the transfer because of insufficient funds.",
    "the card issuer requires 3-D Secure confirmation from the customer.",
    "the payment gateway rejected the transaction because the stored card token has expired. Ask the customer to enter the card details again.",
    "generating the invoice PDF timed out. Retry the export in a few minutes.",
    "the order currency is not supported by the merchant account.",
    "the billing address does not match the card.",
    "the subscription could not be renewed because the plan was archived.",
    "duplicate payment detected; the second charge was blocked.",
]
for code, message in enumerate(error_messages, start=10530):
    knowledge_base[f"err-{code}"] = f"ERR-{code}: {message}"

companies = [
    "ABC Software GmbH, registered office in Munich, managing director Anna Weber",
    "ABC Hardware GmbH, registered office in Augsburg, managing director Jonas Klein",
    "Delta Consulting GmbH, registered office in Munich, managing director Petra Lang",
    "Nord Energie GmbH, registered office in Hamburg, managing director Lukas Brandt",
    "Alpen Bau GmbH, registered office in Rosenheim, managing director Maria Huber",
    "Rhein Logistik GmbH, registered office in Cologne, managing director Tim Becker",
]
for number, company in enumerate(companies, start=80428):
    knowledge_base[f"hrb-{number}"] = f"Commercial register entry HRB {number}: {company}."

docs = [Document(page_content=text, metadata={"id": doc_id}) for doc_id, text in knowledge_base.items()]
len(docs)

22

### BM25 – intuition

BM25 scores a document by summing, for every query term, three ideas:

$$\text{BM25}(q, d) = \sum_{t \in q} \underbrace{\text{IDF}(t)}_{\text{rare = important}} \cdot \underbrace{\frac{f(t,d)\,(k_1+1)}{f(t,d) + k_1\,(1 - b + b\,\frac{|d|}{\text{avgdl}})}}_{\text{term frequency, saturated and length-normalised}}$$

1. **IDF** – a term that appears in few documents (`err-10532`) is worth far more than a common one (`the`).
2. **TF saturation** ($k_1 \approx 1.5$) – the 10th occurrence of a word adds much less than the 1st.
3. **Length normalisation** ($b \approx 0.75$) – a match in a short document counts more than in a long one.

LangChain's `BM25Retriever` wraps `rank-bm25`. The tokenizer matters a lot: the default one splits on spaces only, so we use a regex that keeps identifiers such as `err-10532` or `120/2026` as single tokens.

In [28]:
import re

from langchain_community.retrievers import BM25Retriever


def tokenize(text: str) -> list[str]:
    # keep identifiers like "err-10532" or "120/2026" as single tokens
    return re.findall(r"\w+(?:[-/]\w+)*", text.lower())


print(tokenize("Error ERR-10532 in contract no. 120/2026"))

bm25 = BM25Retriever.from_documents(docs, preprocess_func=tokenize, k=5)

# IDF: rare terms weigh much more than common ones
for term in ["err-10532", "termination", "the"]:
    print(f"IDF({term!r:14}) = {bm25.vectorizer.idf.get(term, 0):.2f}")

['error', 'err-10532', 'in', 'contract', 'no', '120/2026']
IDF('err-10532'   ) = 2.66
IDF('termination' ) = 2.10
IDF('the'         ) = 0.61


### Vector search (FAISS)

In [29]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(docs, embeddings)
vector = vectorstore.as_retriever(search_kwargs={"k": 5})

### Lexical vs semantic – head to head

For each query we know which document is the correct answer. ✅ means the retriever put it in first place.

In [30]:
import pandas as pd

pd.set_option("display.max_colwidth", 80)

# query -> id of the document that answers it
queries = {
    "How do I end my agreement?": "contract-general",
    "Document HRB 80430": "hrb-80430",
    "Problems connecting to the database": "db-troubleshooting",
    "Error ERR-10532": "err-10532",
    "Termination conditions of contract no. 120/2026": "contract-120",
}


def top_ids(retriever, query: str) -> list[str]:
    return [d.metadata["id"] for d in retriever.invoke(query)]


def compare(retrievers: dict, queries: dict[str, str]) -> pd.DataFrame:
    rows = []
    for query, expected in queries.items():
        row = {"query": query}
        for name, retriever in retrievers.items():
            top = top_ids(retriever, query)[0]
            row[name] = ("✅ " if top == expected else "❌ ") + top
        rows.append(row)
    return pd.DataFrame(rows)


compare({"BM25": bm25, "Vector": vector}, queries)

,query,BM25,Vector
0,How do I end my agreement?,✅ contract-general,✅ contract-general
1,Document HRB 80430,✅ hrb-80430,❌ hrb-80428
2,Problems connecting to the database,❌ err-10532,✅ db-troubleshooting
3,Error ERR-10532,✅ err-10532,❌ err-10530
4,Termination conditions of contract no. 120/2026,✅ contract-120,✅ contract-120


### Reciprocal Rank Fusion (RRF)

BM25 scores (0–20+) and cosine similarities (0–1) live on different scales, so we can't just add them. **RRF ignores the scores and uses only the positions** in each ranking:

$$\text{RRF}(d) = \sum_{r \in \text{rankings}} \frac{1}{k + \text{rank}_r(d)}, \qquad k = 60$$

A document that is near the top of *both* lists beats one that is first in only one of them. The constant $k$ dampens the advantage of the very first positions. It is only a few lines of code:

In [31]:
from collections import defaultdict


def reciprocal_rank_fusion(rankings: list[list[str]], k: int = 60) -> list[tuple[str, float]]:
    scores = defaultdict(float)
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking, start=1):
            scores[doc_id] += 1 / (k + rank)
    return sorted(scores.items(), key=lambda item: item[1], reverse=True)


query = "Termination conditions of contract no. 120/2026"
bm25_ranking, vector_ranking = top_ids(bm25, query), top_ids(vector, query)
print("BM25:  ", bm25_ranking)
print("Vector:", vector_ranking)
print("RRF:")
for doc_id, score in reciprocal_rank_fusion([bm25_ranking, vector_ranking]):
    print(f"  {score:.4f}  {doc_id}")

BM25:   ['contract-120', 'contract-121', 'err-10530', 'hrb-80431', 'hrb-80433']
Vector: ['contract-120', 'contract-121', 'contract-general', 'err-10530', 'err-10537']
RRF:
  0.0328  contract-120
  0.0323  contract-121
  0.0315  err-10530
  0.0159  contract-general
  0.0156  hrb-80431
  0.0154  hrb-80433
  0.0154  err-10537


In [32]:
# self-check: a document ranked 2nd in both lists beats one ranked 1st in only one list
assert reciprocal_rank_fusion([["a", "b"], ["c", "b"]])[0][0] == "b"

### Hybrid retriever in LangChain – `EnsembleRetriever`

`EnsembleRetriever` implements exactly this weighted RRF (`c=60` by default). The `weights` let you favour one method, e.g. `[0.3, 0.7]` for natural-language heavy traffic.

In [33]:
from langchain_classic.retrievers import EnsembleRetriever

hybrid = EnsembleRetriever(retrievers=[bm25, vector], weights=[0.5, 0.5])

compare({"BM25": bm25, "Vector": vector, "Hybrid (RRF)": hybrid}, queries)

,query,BM25,Vector,Hybrid (RRF)
0,How do I end my agreement?,✅ contract-general,✅ contract-general,✅ contract-general
1,Document HRB 80430,✅ hrb-80430,❌ hrb-80428,✅ hrb-80430
2,Problems connecting to the database,❌ err-10532,✅ db-troubleshooting,❌ err-10532
3,Error ERR-10532,✅ err-10532,❌ err-10530,✅ err-10532
4,Termination conditions of contract no. 120/2026,✅ contract-120,✅ contract-120,✅ contract-120


**What we see:** vector search confuses near-identical identifiers (`HRB 80428` vs `HRB 80430`), BM25 nails them. Hybrid search fixes the ID queries, but it also inherits BM25's mistake on *Problems connecting to the database*: the right document says *PostgreSQL* and *connection pool*, so BM25 has no strong word overlap with it and ranks a document that merely shares common words. That is the job for the next stage – reranking.

## 2. Reranking with a cross-encoder

The vector search we used is a **bi-encoder**: the query and each document are embedded *independently* and compared with cosine similarity. That is fast (document vectors are precomputed) but coarse – the model never sees the query and the document together.

A **cross-encoder** reads the pair `[query] + [document]` *jointly* in one transformer pass and outputs a relevance score. It is far more accurate, but too slow to run over the whole collection. Hence the two-stage pattern:

```
10 000 documents
       │
       ▼
 Hybrid / vector search   ← fast, high recall
       │
       ▼
  30 candidates
       │
       ▼
    Reranker              ← slow, high precision
       │
       ▼
 5 best chunks
       │
       ▼
      LLM
```

We use `BAAI/bge-reranker-v2-m3` – an open, multilingual (incl. Polish) reranker – through `sentence-transformers`. LangChain's `HuggingFaceCrossEncoder` loads it as a `sentence_transformers.CrossEncoder`, so we load it once and use both APIs.

In [34]:
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from sentence_transformers import CrossEncoder

cross_encoder = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3")
reranker: CrossEncoder = cross_encoder.client  # the underlying sentence-transformers model

reranker.predict([
    ("What is the capital of France?", "Paris is the capital of France."),
    ("What is the capital of France?", "France is famous for its cheese."),
])

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 8285.64it/s]


array([0.99984646, 0.0017077 ], dtype=float32)

### Before vs after reranking

Hybrid search got *Problems connecting to the database* wrong. We take all hybrid candidates for this query and let the cross-encoder reorder them.

In [35]:
query = "Problems connecting to the database"
candidates = hybrid.invoke(query)

scores = reranker.predict([(query, d.page_content) for d in candidates])
before = {d.metadata["id"]: rank for rank, d in enumerate(candidates, start=1)}
reranked = sorted(zip(candidates, scores, strict=True), key=lambda pair: pair[1], reverse=True)

pd.DataFrame(
    {
        "rank after": rank,
        "rank before": before[doc.metadata["id"]],
        "move": f"{before[doc.metadata['id']] - rank:+d}",
        "score": round(float(score), 3),
        "id": doc.metadata["id"],
        "text": doc.page_content,
    }
    for rank, (doc, score) in enumerate(reranked, start=1)
)

,rank after,rank before,move,score,id,text
0,1,3,+2,0.182,db-troubleshooting,"If the application cannot reach PostgreSQL, check whether the connection poo..."
1,2,6,+4,0.000,err-10533,ERR-10533: generating the invoice PDF timed out. Retry the export in a few m...
2,3,5,+2,0.000,db-backup,Database backups run every night at 2:00 and are kept for 30 days.
3,4,1,-3,0.000,err-10532,ERR-10532: the payment gateway rejected the transaction because the stored c...
4,5,7,+2,0.000,err-10535,ERR-10535: the billing address does not match the card.
5,6,4,-2,0.000,password,To reset your password click 'Forgot password' on the login page and follow ...
6,7,2,-5,0.000,err-10530,ERR-10530: the bank declined the transfer because of insufficient funds.
7,8,8,+0,0.000,err-10536,ERR-10536: the subscription could not be renewed because the plan was archived.


### Reranking as a LangChain retriever

`ContextualCompressionRetriever` + `CrossEncoderReranker` turn *any* retriever into a retrieve-then-rerank retriever that returns only the `top_n` best documents.

In [36]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker

hybrid_reranked = ContextualCompressionRetriever(
    base_retriever=hybrid,
    base_compressor=CrossEncoderReranker(model=cross_encoder, top_n=3),
)

for doc in hybrid_reranked.invoke(query):
    print(f"- [{doc.metadata['id']}] {doc.page_content}")

- [db-troubleshooting] If the application cannot reach PostgreSQL, check whether the connection pool is exhausted and whether the firewall allows traffic on port 5432.
- [err-10533] ERR-10533: generating the invoice PDF timed out. Retry the export in a few minutes.
- [db-backup] Database backups run every night at 2:00 and are kept for 30 days.


In [37]:
compare({"Vector": vector, "Hybrid (RRF)": hybrid, "Hybrid + rerank": hybrid_reranked}, queries)

,query,Vector,Hybrid (RRF),Hybrid + rerank
0,How do I end my agreement?,✅ contract-general,✅ contract-general,✅ contract-general
1,Document HRB 80430,❌ hrb-80428,✅ hrb-80430,✅ hrb-80430
2,Problems connecting to the database,✅ db-troubleshooting,❌ err-10532,✅ db-troubleshooting
3,Error ERR-10532,❌ err-10530,✅ err-10532,✅ err-10532
4,Termination conditions of contract no. 120/2026,✅ contract-120,✅ contract-120,✅ contract-120


## 3. Modern chunking strategies

Retrieval can only return what the chunker produced. A chunk that cuts a sentence in half, or that loses the information *which document and section it came from*, cannot be found no matter how good the retriever is.

| Method | How it works |
|---|---|
| Fixed-size chunking | Split every N characters or tokens |
| Recursive chunking | Respect paragraphs, sentences and separators |
| Semantic chunking | Split where the meaning of the text changes |
| Structure-aware chunking | Respect headings, tables and document structure |
| Parent-child retrieval | Search small chunks, return the larger context |
| Contextual Retrieval | Enrich each chunk with information about its source document |

Our sample document is a short annual report in Markdown:

In [38]:
abc_report = """# ABC Corporation – Annual Report 2025

## European market

### Revenue
Revenue grew by 18% compared to the previous year. The growth came mainly from new enterprise contracts and higher subscription prices.

### Employees
The team grew from 420 to 510 people. A new office opened in Kraków.

## Asian market

### Revenue
Revenue grew by 7% compared to the previous year. Currency effects reduced the result by about two percentage points.

### Employees
Headcount stayed at 150 people. Hiring was paused in the second half of the year.

## Outlook
For 2026 the board expects double-digit growth. The main risks are rising cloud costs and stronger competition.
"""

In [39]:
def show(chunks, n: int = 4) -> None:
    for i, chunk in enumerate(chunks[:n], start=1):
        text = chunk.page_content if isinstance(chunk, Document) else chunk
        print(f"--- chunk {i} ({len(text)} chars) ---\n{text}")
    print(f"... {len(chunks)} chunks in total")

### 3.1 Fixed-size chunking

The simplest strategy: cut every N characters. Notice how words, sentences and sections are torn apart.

In [40]:
from langchain_text_splitters import CharacterTextSplitter

fixed = CharacterTextSplitter(separator="", chunk_size=120, chunk_overlap=0)
show(fixed.split_text(abc_report))

--- chunk 1 (120 chars) ---
# ABC Corporation – Annual Report 2025

## European market

### Revenue
Revenue grew by 18% compared to the previous yea
--- chunk 2 (120 chars) ---
r. The growth came mainly from new enterprise contracts and higher subscription prices.

### Employees
The team grew fro
--- chunk 3 (120 chars) ---
m 420 to 510 people. A new office opened in Kraków.

## Asian market

### Revenue
Revenue grew by 7% compared to the pre
--- chunk 4 (120 chars) ---
vious year. Currency effects reduced the result by about two percentage points.

### Employees
Headcount stayed at 150 p
... 6 chunks in total


### 3.2 Recursive chunking

`RecursiveCharacterTextSplitter` tries separators in order – `\n\n`, `\n`, ` `, `""` – and only falls back to a finer one when a piece is still too long. This is the sensible default for plain text. `from_language(Language.MARKDOWN)` / `Language.PYTHON` etc. give it format-specific separators.

In [41]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
show(recursive.split_text(abc_report))

--- chunk 1 (58 chars) ---
# ABC Corporation – Annual Report 2025

## European market
--- chunk 2 (167 chars) ---
## European market

### Revenue
Revenue grew by 18% compared to the previous year. The growth came mainly from new enterprise contracts and higher subscription prices.
--- chunk 3 (99 chars) ---
### Employees
The team grew from 420 to 510 people. A new office opened in Kraków.

## Asian market
--- chunk 4 (146 chars) ---
## Asian market

### Revenue
Revenue grew by 7% compared to the previous year. Currency effects reduced the result by about two percentage points.
... 6 chunks in total


### 3.3 Semantic chunking

Instead of counting characters we look at **meaning**: embed every sentence and start a new chunk where two neighbouring sentences are semantically far apart (here: distance above the 75th percentile). The same algorithm is packaged as `SemanticChunker` in `langchain_experimental`; writing it by hand shows there is no magic.

In [42]:
import numpy as np


def semantic_chunks(text: str, percentile: float = 75) -> list[str]:
    # drop Markdown headings and split into sentences
    plain = " ".join(line for line in text.splitlines() if line and not line.startswith("#"))
    sentences = re.split(r"(?<=[.!?])\s+", plain)
    vectors = np.array(embeddings.embed_documents(sentences))
    vectors /= np.linalg.norm(vectors, axis=1, keepdims=True)
    distances = 1 - np.sum(vectors[:-1] * vectors[1:], axis=1)  # cosine distance to the next sentence
    threshold = np.percentile(distances, percentile)

    chunks, current = [], [sentences[0]]
    for sentence, distance in zip(sentences[1:], distances, strict=True):
        if distance > threshold:
            chunks.append(" ".join(current))
            current = []
        current.append(sentence)
    return chunks + [" ".join(current)]


semantic = semantic_chunks(abc_report)
assert " ".join(semantic).count(".") == abc_report.count(".")  # no sentence lost
show(semantic, n=10)

--- chunk 1 (204 chars) ---
Revenue grew by 18% compared to the previous year. The growth came mainly from new enterprise contracts and higher subscription prices. The team grew from 420 to 510 people. A new office opened in Kraków.
--- chunk 2 (117 chars) ---
Revenue grew by 7% compared to the previous year. Currency effects reduced the result by about two percentage points.
--- chunk 3 (193 chars) ---
Headcount stayed at 150 people. Hiring was paused in the second half of the year. For 2026 the board expects double-digit growth. The main risks are rising cloud costs and stronger competition.
... 3 chunks in total


### 3.4 Structure-aware chunking

Documents already have a structure – headings, sections, tables. `MarkdownHeaderTextSplitter` splits on headings and stores the heading path in **metadata**, so every chunk remembers where it came from (for HTML there is `HTMLHeaderTextSplitter`).

In [43]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "document"), ("##", "section"), ("###", "subsection")]
)
sections = markdown_splitter.split_text(abc_report)

for doc in sections[:3]:
    print(doc.metadata, "\n   ", doc.page_content, "\n")

{'document': 'ABC Corporation – Annual Report 2025', 'section': 'European market', 'subsection': 'Revenue'} 
    Revenue grew by 18% compared to the previous year. The growth came mainly from new enterprise contracts and higher subscription prices. 

{'document': 'ABC Corporation – Annual Report 2025', 'section': 'European market', 'subsection': 'Employees'} 
    The team grew from 420 to 510 people. A new office opened in Kraków. 

{'document': 'ABC Corporation – Annual Report 2025', 'section': 'Asian market', 'subsection': 'Revenue'} 
    Revenue grew by 7% compared to the previous year. Currency effects reduced the result by about two percentage points. 



### 3.5 Parent-child retrieval

Small chunks are **precise to search** (one idea per embedding), large chunks are **useful to read** (the LLM gets enough context). Parent-child retrieval gives you both: it indexes small *child* chunks, but returns their *parent* (here: the whole market section).

In [44]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore
from langchain_core.vectorstores import InMemoryVectorStore

child_vectorstore = InMemoryVectorStore(embeddings)
parent_retriever = ParentDocumentRetriever(
    vectorstore=child_vectorstore,       # small chunks -> embeddings
    docstore=InMemoryStore(),            # parent documents by id
    child_splitter=RecursiveCharacterTextSplitter(chunk_size=80, chunk_overlap=0),
    search_kwargs={"k": 1},
)
# parents: whole "##" market sections (Revenue + Employees together)
parents = MarkdownHeaderTextSplitter(headers_to_split_on=[("#", "document"), ("##", "section")]).split_text(abc_report)
parent_retriever.add_documents(parents)

question = "Where did ABC open a new office?"
print("CHILD found by vector search:\n   ", child_vectorstore.similarity_search(question, k=1)[0].page_content)
print("PARENT returned to the LLM:\n   ", parent_retriever.invoke(question)[0].page_content)

CHILD found by vector search:
    The team grew from 420 to 510 people. A new office opened in Kraków.
PARENT returned to the LLM:
    ### Revenue
Revenue grew by 18% compared to the previous year. The growth came mainly from new enterprise contracts and higher subscription prices.  
### Employees
The team grew from 420 to 510 people. A new office opened in Kraków.


### 3.6 Contextual Retrieval

Look at the two *Revenue* chunks from the report above. Taken out of the document, the chunk

> Revenue grew by 18% compared to the previous year.

does not say **which company, which market or which year**. Add a second report with an identical sentence and neither BM25 nor embeddings can tell them apart.

[Contextual Retrieval](https://www.anthropic.com/news/contextual-retrieval) (Anthropic) fixes this at indexing time: an LLM reads the whole document and writes a short context that is **prepended to every chunk** before it is embedded and indexed by BM25:

> ABC Corporation annual report 2025, European market section: revenue grew by 18% compared to the previous year.

In Anthropic's experiments contextual embeddings + contextual BM25 cut failed retrievals by 49%, and by 67% together with reranking. Prompt caching makes it cheap: the full document is cached once and reused for every chunk.

Let's build a small corpus with **three reports** whose revenue chunks look almost the same:

In [45]:
xyz_report = """# XYZ Retail – Annual Report 2025

## European market

### Revenue
Revenue grew by 11% compared to the previous year. The growth came mainly from online sales.

### Employees
The team shrank from 900 to 860 people after two stores were closed.

## Outlook
For 2026 the board expects stable revenue and lower logistics costs.
"""

abc_report_2024 = """# ABC Corporation – Annual Report 2024

## European market

### Revenue
Revenue grew by 9% compared to the previous year. The growth came mainly from existing customers.

## Asian market

### Revenue
Revenue grew by 12% compared to the previous year. The growth came mainly from the new office in Singapore.
"""

reports = {"ABC-2025": abc_report, "ABC-2024": abc_report_2024, "XYZ-2025": xyz_report}

# structure-aware chunks WITHOUT the headings in the text – the realistic "naive" setup
plain_chunks = []
for report_id, report in reports.items():
    for i, section in enumerate(markdown_splitter.split_text(report)):
        section.metadata |= {"id": f"{report_id}-{i}", "full_document": report}
        plain_chunks.append(section)

len(plain_chunks)

10

Now the contextualisation step – the prompt follows the one published by Anthropic. `llm.batch` sends all chunks in parallel.

In [46]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

context_prompt = ChatPromptTemplate.from_template(
    """<document>
{document}
</document>
Here is the chunk we want to situate within the whole document:
<chunk>
{chunk}
</chunk>
Give a short, succinct context (one sentence: document, company, year, section) that situates this chunk
within the overall document, to improve search retrieval of the chunk.
Answer only with the succinct context and nothing else."""
)
contextualizer = context_prompt | llm | StrOutputParser()

contexts = contextualizer.batch(
    [{"document": c.metadata["full_document"], "chunk": c.page_content} for c in plain_chunks]
)
contextual_chunks = [
    Document(page_content=f"{ctx}\n{c.page_content}", metadata=c.metadata)
    for ctx, c in zip(contexts, plain_chunks, strict=True)
]

print("BEFORE:\n", plain_chunks[0].page_content)
print("\nAFTER:\n", contextual_chunks[0].page_content)

BEFORE:
 Revenue grew by 18% compared to the previous year. The growth came mainly from new enterprise contracts and higher subscription prices.

AFTER:
 In ABC Corporation’s 2025 annual report, the European market revenue section reports 18% growth driven mainly by new enterprise contracts and higher subscription prices.
Revenue grew by 18% compared to the previous year. The growth came mainly from new enterprise contracts and higher subscription prices.


Now the test. We search both indexes for the same question and let the LLM answer **only from the retrieved chunks**. A plain chunk like *"Revenue grew by 18%…"* may even be retrieved, but neither the retriever nor the LLM can tell whether it belongs to ABC or XYZ, to Europe or Asia, to 2024 or 2025.

In [47]:
answer_prompt = ChatPromptTemplate.from_template(
    "Answer only from the CONTEXT. If the CONTEXT does not clearly say it, answer \"I don't know\".\n\n"
    "CONTEXT:\n{context}\n\nQUESTION: {question}"
)
question = "How much did ABC's revenue grow in Europe in 2025?"

for name, chunks in {"PLAIN chunks": plain_chunks, "CONTEXTUAL chunks": contextual_chunks}.items():
    found = FAISS.from_documents(chunks, embeddings).similarity_search(question, k=3)
    context = "\n\n".join(d.page_content for d in found)
    print(f"=== {name} ===")
    for d in found:
        print(f"  [{d.metadata['id']}] {d.page_content[:120]}...")
    print("  LLM:", (answer_prompt | llm | StrOutputParser()).invoke({"context": context, "question": question}), "\n")

=== PLAIN chunks ===
  [ABC-2025-0] Revenue grew by 18% compared to the previous year. The growth came mainly from new enterprise contracts and higher subsc...
  [ABC-2025-2] Revenue grew by 7% compared to the previous year. Currency effects reduced the result by about two percentage points....
  [XYZ-2025-0] Revenue grew by 11% compared to the previous year. The growth came mainly from online sales....
  LLM: I don't know. 

=== CONTEXTUAL chunks ===
  [ABC-2025-0] In ABC Corporation’s 2025 annual report, the European market revenue section reports 18% growth driven mainly by new ent...
  [ABC-2024-0] In ABC Corporation’s 2024 annual report, this passage appears in the European market revenue section.
Revenue grew by 9%...
  [ABC-2025-2] In ABC Corporation’s 2025 annual report, the Asian market revenue section reports 7% growth, partly offset by currency e...
  LLM: ABC’s European revenue grew by 18% in 2025. 



💡 On a corpus this tiny, words such as *ABC* or *2025* appear in most contextual chunks, so their BM25 IDF is close to zero. On real collections with thousands of documents the same context words become strong lexical signals – that is why Anthropic recommends **contextual embeddings + contextual BM25 + reranking** together.

## 4. Everything together – production-style RAG

```
documents ──► structure-aware chunks ──► Contextual Retrieval ──► BM25 + FAISS
                                                                     │
question ──────────────────────────────────────────────► hybrid search (RRF)
                                                                     │
                                                         cross-encoder rerank (top 3)
                                                                     │
                                                                    LLM
```

In [48]:
from langchain_core.runnables import RunnablePassthrough


def build_hybrid(chunks: list[Document], k: int) -> EnsembleRetriever:
    return EnsembleRetriever(
        retrievers=[
            BM25Retriever.from_documents(chunks, preprocess_func=tokenize, k=k),
            FAISS.from_documents(chunks, embeddings).as_retriever(search_kwargs={"k": k}),
        ],
        weights=[0.5, 0.5],
    )


retriever = ContextualCompressionRetriever(
    base_retriever=build_hybrid(contextual_chunks, k=10),  # recall: up to 20 candidates
    base_compressor=CrossEncoderReranker(model=cross_encoder, top_n=3),  # precision
)

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer only from the CONTEXT. Cite the company and section. If the answer is not there, say you don't know.\n\nCONTEXT:\n{context}"),
    ("user", "{question}"),
])


def format_docs(docs: list[Document]) -> str:
    return "\n\n".join(d.page_content for d in docs)


rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

for question in [
    "How much did ABC's revenue grow in Europe in 2025, and why?",
    "Compare European revenue growth of ABC and XYZ in 2025.",
    "How did ABC's European revenue growth change between 2024 and 2025?",
    "What was ABC's revenue in Africa?",
]:
    print(f"Q: {question}\nA: {rag_chain.invoke(question)}\n")

Q: How much did ABC's revenue grow in Europe in 2025, and why?
A: ABC Corporation’s revenue in Europe grew by 18% in 2025, mainly due to new enterprise contracts and higher subscription prices (2025 annual report, European market revenue section).

Q: Compare European revenue growth of ABC and XYZ in 2025.
A: In the European market revenue section of their 2025 annual reports, ABC Corporation reported 18% growth, while XYZ Retail reported 11% growth. ABC’s growth was 7 percentage points higher. ABC attributed growth mainly to new enterprise contracts and higher subscription prices; XYZ attributed it mainly to online sales.

Q: How did ABC's European revenue growth change between 2024 and 2025?
A: ABC Corporation’s European revenue growth increased from 9% in 2024 to 18% in 2025—up 9 percentage points. The 2024 growth came mainly from existing customers; in 2025, it was driven mainly by new enterprise contracts and higher subscription prices. (ABC Corporation, European market revenue se